In [37]:
import random

import torch
torch.manual_seed(42)
import numpy as np 
import pandas as pd

device = "cuda" 

# Import texts and embedding df
text_chunks_and_embedding_df = pd.read_csv("all_embeddings_df.csv")

text_chunks_and_embedding_df.head()

,page_number,sentence_chunk,chunk_char_count,chunk_word_count,chunk_token_count,embedding,product
0,1,ViPNet Coordinator HW 5.Подготовка к работе | ...,1274,159,318.5,"[0.0060768756084144115, -0.042224615812301636,...",подготовка
1,1,NAT позволяет решить две задачи:  Подключение...,930,112,232.5,"[-0.007902384735643864, -0.02230028621852398, ...",подготовка
2,2,ViPNet Coordinator HW 5.Подготовка к работе | ...,234,35,58.5,"[0.008719977922737598, -0.03479960188269615, -...",подготовка
3,3,ViPNet Coordinator HW 5.Подготовка к работе | ...,1222,149,305.5,"[-0.022762130945920944, -0.0357675738632679, 0...",подготовка
4,3,Событие срабатывания правила IPS регистрируетс...,314,43,78.5,"[-0.017361363396048546, 0.0026617932599037886,...",подготовка


In [29]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import pymorphy3

In [30]:
import nltk
nltk.download('punkt')
from nltk.tokenize import sent_tokenize
nltk.download('all')


[nltk_data] Downloading package punkt to /home/kirill/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading collection 'all'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to /home/kirill/nltk_data...
[nltk_data]    |   Package abc is already up-to-date!
[nltk_data]    | Downloading package alpino to
[nltk_data]    |     /home/kirill/nltk_data...
[nltk_data]    |   Package alpino is already up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     /home/kirill/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger is already up-
[nltk_data]    |       to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_eng to
[nltk_data]    |     /home/kirill/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger_eng is already
[nltk_data]    |       up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_ru to
[nltk_data]    |     /home/kirill/nltk_da

True

In [31]:
nltk.download('punkt')
from nltk.tokenize import word_tokenize
lemmatizer = nltk.stem.WordNetLemmatizer()

[nltk_data] Downloading package punkt to /home/kirill/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [ ]:
from sentence_transformers import util, SentenceTransformer

embedding_model = SentenceTransformer(model_name_or_path="ai-forever/ru-en-RoSBERTa", 
                                      device=device) # choose the device to load the model to

/home/kirill/anaconda3/envs/testtf/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-01-28 17:38:06.742946: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Some weights of RobertaModel were not initialized from the model checkpoint at ai-forever/ru-en-RoSBERTa and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [25]:
from transformers import AutoTokenizer, AutoModelForCausalLM


model_name = "t-tech/T-lite-it-2.1"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto",
)

`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 4/4 [00:31<00:00,  7.99s/it]


In [41]:
def retrieve_relevant_resources(query: str,
                                embeddings: torch.tensor,
                                model: SentenceTransformer=embedding_model,
                                n_resources_to_return: int=5):

    # Embed the query
    query_embedding = model.encode(query, 
                                   convert_to_tensor=True) 
    
    #Here
    text_chunks_and_embedding_df = pd.read_csv("all_embeddings_df.csv")
    tokens=word_tokenize(query.lower())
    lemmatized_words = [lemmatizer.lemmatize(word) for word in tokens]
    products = text_chunks_and_embedding_df['product'].unique().tolist()
    f=[]
    for i in lemmatized_words:
        for j in products:
            if i == j:
                f.append(j)
    if not f:
        df_temp = text_chunks_and_embedding_df
        df_temp["embedding"] = df_temp["embedding"].apply(lambda x: np.fromstring(x.strip("[]"), sep=", "))
        pages_and_chunks = df_temp.to_dict(orient="records")
        embeddings = torch.tensor(np.array(df_temp["embedding"].tolist()), dtype=torch.float32).to(device)
        embeddings.shape
    else:
        df_temp = text_chunks_and_embedding_df[text_chunks_and_embedding_df['product'].isin(f)]
        df_temp["embedding"] = df_temp["embedding"].apply(lambda x: np.fromstring(x.strip("[]"), sep=", "))
        pages_and_chunks = df_temp.to_dict(orient="records")
        embeddings = torch.tensor(np.array(df_temp["embedding"].tolist()), dtype=torch.float32).to(device)
        embeddings.shape
    
    # Get dot product scores on embeddings
    dot_scores = util.cos_sim(query_embedding, embeddings)[0]
    

    scores, indices = torch.topk(input=dot_scores, 
                                 k=n_resources_to_return)

    return scores, indices, pages_and_chunks

In [43]:
retrieve_relevant_resources(query="Выбор профиля производительности?",embeddings=embeddings)


(tensor([0.7211, 0.6701, 0.6684, 0.6500, 0.6228], device='cuda:0'),
 tensor([1125, 1124,  508, 1126,   38], device='cuda:0'),
 [{'page_number': 1,
   'sentence_chunk': 'ViPNet Coordinator HW 5.Подготовка к работе | 15 Межсетевой экран В межсетевом экране ViPNet Coordinator HW реализованы следующие механизмы информационной безопасности, присущие межсетевым экранам нового поколения: \uf0b7 фильтрация трафика на сетевом и транспортном уровнях модели OSI с контролем состояния сессий; \uf0b7 расширенная инспекция трафика (Deep Packet Inspection) с целью отслеживания активности приложений и прикладных протоколов; \uf0b7 возможность определения политик безопасности на уровне пользователей служб каталогов Active Directory или LDAP.  Примечание. ViPNet Coordinator HW классифицирует каждую сессию по типу прикладного протокола один раз до истечения времени жизни сессии.При изменении списка правил классификация сессий по прикладному протоколу заново не проводится. С помощью сетевых фильтров можно 

In [44]:
retrieve_relevant_resources(query="Выбор профиля web производительности?",embeddings=embeddings)

(tensor([0.6283, 0.6264, 0.6078, 0.5753, 0.5572], device='cuda:0'),
 tensor([293, 292, 294, 434, 436], device='cuda:0'),
 [{'page_number': 1,
   'sentence_chunk': 'ViPNet Coordinator HW 5.Настройка с помощью веб-интерфейса | 15 Подключение к ViPNet Coordinator HW По умолчанию подключение к ViPNet Coordinator HW с помощью веб-интерфейса запрещено.Если вы не разрешили подключение при инициализации устройства, сделайте это (см. «Подготовка к работе», раздел «После инициализации»). Подключение с помощью веб-интерфейса следует выполнять с защищенных узлов ViPNet, связанных с ViPNet Coordinator HW.Возможность подключения пользователей зависит от способа аутентификации: \uf0b7 Аутентификация по паролю — разрешено для локальных (user и admin) и централизованных пользователей. \uf0b7 Аутентификация по сертификату — разрешено для локальных пользователей user и admin с паролем их учётных записей (USB-токен при подключении не требуется) и для централизованных пользователей. По умолчанию для подклю

In [45]:
def prompt_formatter(query: str, 
                     context_items: list[dict]) -> str:
    """
    Augments query with text-based context from context_items.
    """
    context = "- " + "\n- ".join([item["sentence_chunk"] for item in context_items])

    base_prompt = '''Ты  виртуальный ассистент. Твоя задача — быть полезным диалоговым ассистентом. Дайте себе время подумать, извлекая соответствующие отрывки из контекста, прежде чем отвечать на вопрос.
Не повторяйте мысль, а только формулируйте ответ.
Убедитесь, что ваши ответы максимально понятны.
Используйте следующие примеры в качестве ориентира для выбора идеального ответа.
\nПример 1:
Вопрос: Что такое жирорастворимые витамины?
Ответ: К жирорастворимым витаминам относятся витамин А, витамин D, витамин Е и витамин К. Эти витамины усваиваются вместе с жирами, поступающими с пищей, и могут накапливаться в жировой ткани организма и печени для последующего использования. Витамин А важен для зрения, иммунной системы и здоровья кожи. Витамин D играет важную роль в усвоении кальция и укреплении костей. Витамин Е действует как антиоксидант, защищая клетки от повреждений. Витамин К необходим для свертывания крови и метаболизма костной ткани.
\nПример 2:
Вопрос: Каковы причины развития сахарного диабета 2 типа?
Ответ: Сахарный диабет 2 типа часто ассоциируется с перееданием, в частности, с чрезмерным потреблением калорий, что приводит к ожирению. К таким факторам относится диета с высоким содержанием рафинированного сахара и насыщенных жиров, что может привести к резистентности к инсулину - состоянию, при котором клетки организма неэффективно реагируют на инсулин. Со временем поджелудочная железа перестает вырабатывать достаточное количество инсулина для поддержания уровня сахара в крови, что приводит к развитию диабета 2 типа. Кроме того, чрезмерное потребление калорий без достаточной физической активности увеличивает риск, способствуя увеличению веса и накоплению жира, особенно в области живота, что еще больше способствует резистентности к инсулину.
\nПример 3:
Вопрос: Насколько важно увлажнение для физической работоспособности?
Ответ: Гидратация имеет решающее значение для физической работоспособности, поскольку вода играет ключевую роль в поддержании объема циркулирующей крови, регулировании температуры тела и обеспечении доставки питательных веществ и кислорода к клеткам. Адекватная гидратация необходима для оптимального функционирования мышц, выносливости и восстановления. Обезвоживание может привести к снижению работоспособности, усталости и повышенному риску заболеваний, связанных с жарой, таких как тепловой удар. Употребление достаточного количества воды до, во время и после тренировки помогает обеспечить максимальную физическую работоспособность и восстановление.
\nТеперь используйте следующие элементы контекста для ответа на запрос пользователя:
{context}
'''


    base_prompt = base_prompt.format(context=context, query=query)
    dialogue_template = [
        {
        "role": "system",
        "content": base_prompt
    },
        {"role": "user",
        "content": query}
    ]

    prompt = tokenizer.apply_chat_template(conversation=dialogue_template,
                                          tokenize=False,
                                          add_generation_prompt=True)
    return prompt

In [46]:
def ask(query, 
        temperature=0.5,
        max_new_tokens=1024,
        format_answer_text=True, 
        return_answer_only=True):
    scores, indices, pages_and_chunks = retrieve_relevant_resources(query=query,
                                                  embeddings=embeddings)
    
    context_items = [pages_and_chunks[i] for i in indices]

    for i, item in enumerate(context_items):
        item["score"] = scores[i].cpu() 
        
    prompt = prompt_formatter(query=query,
                              context_items=context_items)
    
    input_ids = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(**input_ids,
                                 temperature=temperature,
                                 do_sample=True,
                                 max_new_tokens=max_new_tokens)
    
    output_text = tokenizer.decode(outputs[0])

    if format_answer_text:
        # Replace special tokens and unnecessary help message
        output_text = output_text.replace(prompt, "").replace("<|im_end|>", "")

    # Only return the answer without the context items
    if return_answer_only:
        return output_text
    
    return output_text, context_items

In [47]:
def answer_question(input_question:str):
    answer, context_items = ask(query=input_question, 
                                temperature=0.5,
                                max_new_tokens=1024,
                                return_answer_only=False)


    print(f"Аугментированные вводные:")
    print(context_items)
    print(f"Ответ:\n")
    print(answer)

In [ ]:
user_question= "Событие срабатывания правила IPS"
answer_question(user_question)

Аугментированные вводные:
[{'page_number': 3, 'sentence_chunk': 'Событие срабатывания правила IPS регистрируется в журнале IP-пакетов. Средство IPS выявляет и предотвращает: \uf0b7 Попытки эксплуатации уязвимостей в ПО объектов защищаемой сети. \uf0b7 Атаки на сетевые службы и серверы. \uf0b7 Атаки типа «отказ в обслуживании» (DoS-атаки). \uf0b7 Аномальный IP-трафик. \uf0b7 Сетевую активность вирусов.', 'chunk_char_count': 314, 'chunk_word_count': 43, 'chunk_token_count': 78.5, 'embedding': array([-0.01736136,  0.00266179,  0.01511953, ..., -0.02409583,
        0.02487063, -0.00275595], shape=(1024,)), 'product': 'подготовка', 'score': tensor(0.7826)}, {'page_number': 134, 'sentence_chunk': 'ViPNet Coordinator HW 5.Настройка с помощью веб-интерфейса | 148  Рисунок 81.Настройка правила IPS 3 На панели Настраиваемые параметры вы можете: o Изменить статус правила IPS. o Изменить действие при срабатывании правила IPS.  Примечание.Если для правила IPS по умолчанию задано действие Предупрежд

In [48]:
user_question= "Событие срабатывания правила IPS web"
answer_question(user_question)

Аугментированные вводные:
[{'page_number': 134, 'sentence_chunk': 'ViPNet Coordinator HW 5.Настройка с помощью веб-интерфейса | 148  Рисунок 81.Настройка правила IPS 3 На панели Настраиваемые параметры вы можете: o Изменить статус правила IPS. o Изменить действие при срабатывании правила IPS.  Примечание.Если для правила IPS по умолчанию задано действие Предупреждать, то его нельзя изменить на действие Блокировать. o В поле Комментарий добавить примечание. 4 Нажмите Применить. 5 Нажмите . Журнал событий для выбранного правила IPS соответствует журналу регистрации IP-пакетов, отфильтрованному по выбранному правилу.', 'chunk_char_count': 554, 'chunk_word_count': 76, 'chunk_token_count': 138.5, 'embedding': array([-0.00818202, -0.01606352,  0.02561151, ..., -0.00560405,
        0.04105454,  0.00174033], shape=(1024,)), 'product': 'web', 'score': tensor(0.7278)}, {'page_number': 134, 'sentence_chunk': 'Для его просмотра нажмите  в строке выбранного правила.  Групповые операции с правилами 